# Sesión 2. Feature engineering, validación temporal y backtesting

**Curso: Pronósticos Macroeconómicos con Machine Learning**

**Autor**: [Renato Vassallo](https://renatovassallo.github.io)

En esta sesión construiremos un pipeline completo para pronosticar la inflación peruana a un mes. Al terminar podremos responder cinco preguntas:

1. ¿Qué información estaba realmente disponible en cada origen?
2. ¿Cómo se separan train, validation y test sin viajar en el tiempo?
3. ¿Qué hacen Ridge y Lasso cuando mezclamos señal candidata, redundancia y ruido conocido?
4. ¿Un modelo supera a benchmarks simples fuera de muestra?
5. ¿Qué variables movieron una predicción concreta?

> **La idea que guía la sesión**: la arquitectura del ejercicio (información comparable, validación temporal y benchmarks honestos) importa más que cambiar de algoritmo.

In [ ]:
from IPython.display import display
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------
# Base de la sesión: session2/ es autocontenida (utils.py + data/)
# ---------------------------------------------------------------------

# Funciona abriendo el notebook desde session2/ o desde la raíz del repo.
candidatos = [Path.cwd() / "session2", Path.cwd(), *Path.cwd().parents]
BASE_DIR = next(
    (
        ruta.resolve()
        for ruta in candidatos
        if (ruta / "utils.py").exists() and (ruta / "data" / "monthly.csv").exists()
    ),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError(
        "No se encontró utils.py junto a data/monthly.csv en la carpeta de la sesión."
    )

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import utils as U

U.set_style()   # utils.py resuelve data/ y figures/ desde su propia carpeta

# ---------------------------------------------------------------------
# Parámetros generales
# ---------------------------------------------------------------------

HORIZON = 1
TEST_START = pd.Timestamp("2021-01-01")
RANDOM_STATE = 42

MODEL_ORDER = ["RW", "AR(3)-OLS", "Ridge", "Lasso", "RandomForest"]
PRIMARY_MODEL = "Ridge"
REGULARIZED_MODELS = ["Ridge", "Lasso"]

GDP_EXTRA = [
    "g_circ",
    "g_us_indpro",
    "g_pbim_yoy",
    "g_credmn",
]

DRIVER_COLUMNS = [
    "g_tc",
    "g_p_wti",
    "g_ipm",
    "int",
    "g_emiprim",
    "g_circ",
    "g_us_indpro",
    "g_pbim_yoy",
    "g_credmn",
]

SUBPERIODS_PRESET = {
    "Test completo": (TEST_START, None),
    "Inflación alta 2021-2023": (TEST_START, pd.Timestamp("2023-12-01")),
    "Normalización 2024-2026": (pd.Timestamp("2024-01-01"), None),
}

print(f"Sesión: {BASE_DIR.name}")
print(f"Datos: {U.DATA}")
print(f"Figuras: {U.FIGS}")

In [ ]:
raw = U.load_monthly().sort_index()
infl = raw[U.INFLATION_COLUMNS].copy()
meta = U.load_metadata()

# Controles mínimos antes de modelar
assert raw.index.is_monotonic_increasing
assert raw.index.is_unique
assert raw.index.equals(infl.index)
assert raw.index.equals(pd.date_range(raw.index.min(), raw.index.max(), freq="MS"))
assert {"ipc_yoy", "ipc_mom", *DRIVER_COLUMNS}.issubset(raw.columns)

print(f"Muestra mensual: {raw.index.min():%Y-%m} a {raw.index.max():%Y-%m}")
print(f"Observaciones: {len(raw)} | Series disponibles: {raw.shape[1]}")
print(f"Panel consolidado: inflación y drivers macro en {U.DATA / 'monthly.csv'}")
display(raw[["ipc_yoy", "ipc_mom", *GDP_EXTRA]].tail(3).round(2))


## 1. Primero definimos la pregunta de pronóstico

El origen es el momento en que se publica el IPC del mes \(t\), tres días después de terminar ese mes. Queremos pronosticar la inflación interanual del mes siguiente:

$$\text{información disponible en }t \quad \longrightarrow \quad \widehat{\pi}^{12m}_{t+1}. $$

Por tanto, todos los modelos pueden observar \(\pi^{12m}_t\). El RW usa esa información y los demás modelos también deben recibirla. Para que la penalización tenga un ancla económica, Ridge, Lasso y Random Forest pronosticarán la **corrección al RW**:

$$\Delta_{t+1}=\pi^{12m}_{t+1}-\pi^{12m}_t, \qquad \widehat{\pi}^{12m}_{t+1}=\pi^{12m}_t+\widehat{\Delta}_{t+1}. $$

Una corrección igual a cero reproduce exactamente el RW. Esta parametrización permite leer a los challengers como desviaciones respecto del benchmark. La penalización contrae hacia cero la parte explicada por las features; como el intercepto no se penaliza, el límite no es exactamente el RW sino una corrección constante. Esto no cambia la pregunta ni la métrica: el error de \(\widehat{\Delta}_{t+1}\) es el mismo error del pronóstico reconstruido en nivel. El AR(3)-OLS se mantiene como benchmark lineal directo.

## 2. Calendario de publicación

Usaremos una convención fija y visible. Si una serie se publica como máximo tres días después del cierre del mes, usamos su valor de \(t\). Si demora más, usamos el último mes que ya estaría publicado. Esta es una aproximación mensual de pseudo-tiempo-real, no una base de vintages. Por ello controla el calendario de publicación, pero no elimina el sesgo potencial de usar datos históricos revisados, especialmente en series de actividad.

In [ ]:
PUBLICATION_LAGS = {
    "ipc_yoy": 0,       # publicado en el origen
    "ipc_mom": 0,       # publicado junto con ipc_yoy
    "g_p_wti": 0,       # retraso de 3 días
    "g_tc": 1,          # retraso de 10 días
    "int": 1,           # retraso de 10 días
    "g_emiprim": 1,     # retraso de 10 días
    "g_us_indpro": 1,   # retraso de 15 días
    "g_credmn": 1,      # retraso de 22 días
    "g_ipm": 2,         # retraso de 40 días
    "g_circ": 2,        # retraso de 40 días
    "g_pbim_yoy": 2,    # retraso de 51 días
}

SERIES_LABELS = {
    "ipc_yoy": "inflación interanual",
    "ipc_mom": "inflación mensual",
    "g_tc": "tipo de cambio",
    "g_p_wti": "precio WTI",
    "g_ipm": "precios de importación",
    "int": "tasa interbancaria",
    "g_emiprim": "emisión primaria",
    "g_circ": "circulante",
    "g_us_indpro": "producción industrial de EE.UU.",
    "g_pbim_yoy": "PBI mensual interanual",
    "g_credmn": "crédito en soles",
}

calendar = pd.DataFrame({
    "serie": pd.Series(SERIES_LABELS),
    "retraso_días": meta.loc[list(PUBLICATION_LAGS), "delay_days"],
    "lag_meses_usado": pd.Series(PUBLICATION_LAGS),
    "transformación": {
        column: ("sin transformación" if column.startswith("ipc_")
                 else "nivel" if column == "int" else "signed-log")
        for column in PUBLICATION_LAGS
    },
}).loc[list(PUBLICATION_LAGS)]
calendar.index.name = "columna"
display(calendar)

## 3. Features: cada columna debe tener una historia

La biblioteca amplia tiene tres familias:

1. **Dinámica de inflación**: seis cambios mensuales pasados de la inflación interanual.
2. **Presión reciente**: seis rezagos de inflación mensual y medias anualizadas de 3 y 6 meses.
3. **Drivers macroeconómicos**: nueve indicadores, cada uno representado por último dato disponible, media de 3 meses y cambio frente a tres meses atrás. El panel mensual unificado reúne inflación, circulante, producción industrial de EE.UU., PBI mensual y crédito en una sola tabla.

Primero aplicamos el lag de publicación y después construimos media y cambio. Salvo la tasa interbancaria, comprimimos valores extremos con `signed_log(x) = sign(x) * log(1 + abs(x))`. Las representaciones vecinas son deliberadamente redundantes: ese es el problema para el que Ridge y Lasso resultan útiles. Un indicador económico que Lasso lleve a cero sigue siendo una señal candidata, no ruido verdadero.

In [ ]:
def signed_log(values):
    """Transformación fija, simétrica y definida también para valores negativos."""
    return np.sign(values) * np.log1p(np.abs(values))


panel = pd.DataFrame(index=raw.index)
panel.index.name = "origin_month"

# Ancla del RW y features exclusivas del benchmark AR(3).
panel["y_t"] = raw["ipc_yoy"]
panel["y_tm1"] = raw["ipc_yoy"].shift(1)
panel["y_tm2"] = raw["ipc_yoy"].shift(2)
AR_FEATURES = ["y_t", "y_tm1", "y_tm2"]

# 1) Seis cambios mensuales ya observados de la inflación interanual.
DYNAMIC_FEATURES = []
for lag in range(1, 7):
    name = f"delta_y_lag{lag}"
    panel[name] = raw["ipc_yoy"].shift(lag - 1) - raw["ipc_yoy"].shift(lag)
    DYNAMIC_FEATURES.append(name)

# 2) Inflación mensual reciente y dos resúmenes solapados.
MOMENTUM_FEATURES = []
for lag in range(6):
    name = f"ipc_mom_lag{lag}"
    panel[name] = raw["ipc_mom"].shift(lag)
    MOMENTUM_FEATURES.append(name)
for window in [3, 6]:
    name = f"ipc_mom_ma{window}_ann"
    panel[name] = raw["ipc_mom"].rolling(window).mean() * 12
    MOMENTUM_FEATURES.append(name)

# 3) Disponibilidad primero; transformación y dinámica después.
DRIVER_FEATURES = []
for column in DRIVER_COLUMNS:
    available = raw[column].shift(PUBLICATION_LAGS[column])
    transformed = available if column == "int" else signed_log(available)
    names = [f"{column}_latest", f"{column}_ma3", f"{column}_delta3"]
    panel[names[0]] = transformed
    panel[names[1]] = transformed.rolling(3).mean()
    panel[names[2]] = transformed - transformed.shift(3)
    DRIVER_FEATURES.extend(names)


FEATURE_GROUPS = {
    "Dinámica de inflación": DYNAMIC_FEATURES,
    "Presión mensual": MOMENTUM_FEATURES,
    "Drivers macro": DRIVER_FEATURES,
}
ALL_FEATURES = [feature for features in FEATURE_GROUPS.values() for feature in features]
FEATURE_TO_GROUP = {feature: group for group, features in FEATURE_GROUPS.items() for feature in features}
ECONOMIC_FEATURES = [feature for feature in ALL_FEATURES]

assert [len(DYNAMIC_FEATURES), len(MOMENTUM_FEATURES), len(DRIVER_FEATURES)] == [6, 8, 27]
assert len(ALL_FEATURES) == 41

# La fila t pronostica t+1; delta_next es la corrección observada al RW.
panel["y_next"] = raw["ipc_yoy"].shift(-HORIZON)
panel["delta_next"] = panel["y_next"] - panel["y_t"]
panel["target_date"] = panel.index + pd.offsets.MonthBegin(HORIZON)
panel["release_date"] = (
    panel.index + pd.offsets.MonthEnd(0)
    + pd.Timedelta(days=int(meta.loc["ipc_yoy", "delay_days"]))
)

# Una muestra común para que todos los challengers vean los mismos meses.
panel = panel.dropna(subset=AR_FEATURES + ALL_FEATURES + ["y_next", "delta_next"])

feature_summary = pd.DataFrame([
    {
        "familia": group,
        "n_features": len(features),
        "ejemplos": ", ".join(features[:3]) + (", ..." if len(features) > 3 else ""),
    }
    for group, features in FEATURE_GROUPS.items()
]).set_index("familia")
print(f"Panel final: {len(panel)} meses x {len(ALL_FEATURES)} features para challengers")
display(feature_summary)
display(panel[["target_date", "y_t", "delta_next", "y_next"]].tail(3).round(3))

# La transformación reduce el leverage de extremos sin mirar el objetivo.
wti_raw = raw.loc["2018":, "g_p_wti"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
axes[0].plot(wti_raw.index, wti_raw, color=U.MUTED, lw=1.1)
axes[0].set_title("WTI disponible: escala original", loc="left", fontsize=10)
axes[0].set_ylabel("valor original del CSV")
axes[1].plot(wti_raw.index, signed_log(wti_raw), color=U.ACCENT, lw=1.1)
axes[1].set_title("La misma serie después de signed-log", loc="left", fontsize=10)
axes[1].set_ylabel("signed-log")
U.save_fig(fig, "B01_transformacion_wti")
plt.show()

## 4. Train, validation y test sin viajar en el tiempo

Primero bloqueamos el test. Ninguna feature, hiperparámetro o modelo se elegirá con resultados posteriores a 2020.

- **Train inicial**: objetivos hasta 2015.
- **Validation**: cinco folds anuales expansivos, de 2016 a 2020.
- **Test**: objetivos de enero de 2021 en adelante. Los subperiodos 2021-2023 y 2024-2026 también quedan declarados antes de estimar.

En el test reestimaremos coeficientes cada mes con toda la información pasada. Eso es válido: las realizaciones anteriores pueden usarse cuando ya fueron publicadas. Lo prohibido es usar el test para cambiar features, hiperparámetros o la configuración de los modelos.

> **Una confesión honesta:** el bloqueo es correcto dentro del ejercicio, pero esta notebook fue diseñada cuando 2021-2026 ya era historia observada. Sus cifras son demostrativas, no evidencia confirmatoria de un test genuinamente nuevo.

In [ ]:
development = panel[panel["target_date"] < TEST_START].copy()
test = panel[panel["target_date"] >= TEST_START].copy()

assert development["target_date"].max() < test["target_date"].min()

tscv = TimeSeriesSplit(n_splits=5, test_size=12, gap=HORIZON - 1)
CV_SPLITS = list(tscv.split(development))

print("Bloques bloqueados antes de estimar:")
print(f"  desarrollo: {development.target_date.min():%Y-%m} a {development.target_date.max():%Y-%m}")
print(f"  test:       {test.target_date.min():%Y-%m} a {test.target_date.max():%Y-%m}")

fig, ax = plt.subplots(figsize=(10, 4.3))
for fold, (idx_train, idx_valid) in enumerate(CV_SPLITS, start=1):
    train_dates = development.iloc[idx_train]["target_date"]
    valid_dates = development.iloc[idx_valid]["target_date"]
    ax.plot(train_dates, [fold] * len(train_dates), lw=8, color=U.TINT,
            solid_capstyle="butt", label="train" if fold == 1 else None)
    ax.plot(valid_dates, [fold] * len(valid_dates), lw=8, color=U.ACCENT,
            solid_capstyle="butt", label="validation" if fold == 1 else None)

test_row = len(CV_SPLITS) + 1
ax.plot(test["target_date"], [test_row] * len(test), lw=8, color=U.BLUE,
        solid_capstyle="butt", label="test bloqueado")
ax.axvline(TEST_START, color=U.INK, lw=0.9, ls="dashed")
ax.set_yticks([1, 2, 3, 4, 5, test_row])
ax.set_yticklabels([f"fold {i}" for i in range(1, 6)] + ["test"])
ax.set_xlabel("fecha del objetivo")
ax.set_title("El test se bloquea antes de elegir el modelo", loc="left")
ax.legend(ncol=3, fontsize=9)
ax.grid(False)
U.save_fig(fig, "B02_particion_cv_test")
plt.show()

### ¿Qué tan distintos son los tres bloques?

Antes de estimar, conviene mirar la variable objetivo. El test post-2020 contiene el episodio de mayor inflación de la muestra reciente y luego su normalización. Esta diferencia de distribución hace que el ejercicio sea exigente y explica por qué un ranking aprendido antes de 2021 puede no ser estable después.

En la figura, `validation 2016-2020` representa la unión de los cinco folds de validación. Como el esquema es expansivo, un año usado como validation en un fold pasa a formar parte del train de folds posteriores.

Las familias de modelos, los grids, el modelo principal y los subperiodos ya quedaron declarados en la configuración inicial. Esta inspección describe el cambio de distribución; no autoriza a revisar esas decisiones.

In [ ]:
TRAIN_SAMPLE_START = panel["target_date"].min()
SAMPLE_BLOCKS = {
    "Train inicial": (TRAIN_SAMPLE_START, pd.Timestamp("2015-12-01")),
    "Validation 2016-2020": (pd.Timestamp("2016-01-01"), pd.Timestamp("2020-12-01")),
    "Test 2021+": (TEST_START, panel["target_date"].max()),
}

fig, ax = plt.subplots(figsize=(10, 4.3))
inflation_plot = infl.loc["2000-01-01":, "ipc_yoy"]
ax.plot(inflation_plot.index, inflation_plot, color=U.INK, lw=1.4,
        label="inflación observada")
ax.axvspan(TRAIN_SAMPLE_START, pd.Timestamp("2015-12-31"),
           color=U.TINT, alpha=0.65, label="train")
ax.axvspan(pd.Timestamp("2016-01-01"), pd.Timestamp("2020-12-31"),
           color=U.GOLD, alpha=0.16, label="validation")
ax.axvspan(TEST_START, inflation_plot.index.max(),
           color=U.BLUE, alpha=0.10, label="test")
ax.axhspan(1, 3, color=U.OLIVE, alpha=0.08, label="rango meta")
ax.set_ylabel("inflación, % a/a")
ax.set_title("Inflación y partición temporal del ejercicio", loc="left")
ax.legend(fontsize=9, ncol=3)
U.save_fig(fig, "B03_inflacion_train_validation_test")
plt.show()

block_rows = []
for name, (start, end) in SAMPLE_BLOCKS.items():
    values = panel.loc[panel["target_date"].between(start, end), "y_next"]
    block_rows.append({
        "subconjunto": name, "n": len(values),
        "promedio": values.mean(), "desviación estándar": values.std(),
    })

sample_summary = pd.DataFrame(block_rows).set_index("subconjunto")
display(sample_summary.round(3))

## 5. Hiperparámetros dentro de development

Ridge y Lasso necesitan features comparables. `StandardScaler` y el modelo viven en un mismo `Pipeline`, por lo que el escalador se estima únicamente con el train de cada fold. Ambos aprenden `delta_next`, la corrección al RW. Como el ancla `y_t` es común, el RMSE de esa corrección coincide exactamente con el RMSE del pronóstico reconstruido.

El grid de Ridge incluye `alpha=0`, de modo que validation puede escoger el límite no penalizado. Como las medias móviles son combinaciones exactas de rezagos ya incluidos, usamos el solver SVD para evaluar ese punto sin invertir una matriz singular. AR(3)-OLS conserva el papel de benchmark lineal parsimonioso.

Para Random Forest probamos un grid pequeño de profundidad, tamaño mínimo de hoja y proporción de features. El número de árboles queda fijo. Después del tuning mostraremos la magnitud asignada por Ridge y cuántas variables llevó Lasso exactamente a cero. Esto es un diagnóstico de contracción y sparsity, no una prueba de que Lasso haya identificado causalidad.

Validation elige los hiperparámetros de los tres modelos. No usamos estos folds para organizar una carrera adicional ni para seleccionar un modelo ganador. El test todavía no participa.

In [ ]:
RIDGE_ALPHA_GRID = np.r_[0.0, np.logspace(-2, 4, 37)]
LASSO_ALPHA_GRID = np.logspace(-4, 0, 33)
RF_PARAM_GRID = {
    "max_depth": [3, None],
    "min_samples_leaf": [3, 8],
    "max_features": [0.6, 1.0],
}

SEARCH_SPACES = {
    "Ridge": (
        make_pipeline(StandardScaler(), Ridge(solver="svd")),
        {"ridge__alpha": RIDGE_ALPHA_GRID},
    ),
    "Lasso": (
        make_pipeline(StandardScaler(), Lasso(max_iter=20_000)),
        {"lasso__alpha": LASSO_ALPHA_GRID},
    ),
    "RandomForest": (
        RandomForestRegressor(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1,
        ),
        RF_PARAM_GRID,
    ),
}

TUNED_MODELS = {}
TUNING_RESULTS = {}
BEST_PARAMS = {}

for name, (estimator, grid) in SEARCH_SPACES.items():
    search = GridSearchCV(
        estimator, grid, cv=CV_SPLITS,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
    )
    search.fit(development[ALL_FEATURES], development["delta_next"])
    TUNED_MODELS[name] = clone(search.best_estimator_)
    TUNING_RESULTS[name] = pd.DataFrame(search.cv_results_)
    BEST_PARAMS[name] = {
        key: value.item() if isinstance(value, np.generic) else value
        for key, value in search.best_params_.items()
    }
    print(f"{name:12s} | RMSE medio de validation = {-search.best_score_:.3f}")
    print(f"  hiperparámetros: {BEST_PARAMS[name]}")

RIDGE_ALPHA_STAR = float(BEST_PARAMS["Ridge"]["ridge__alpha"])
LASSO_ALPHA_STAR = float(BEST_PARAMS["Lasso"]["lasso__alpha"])
RIDGE_DISPLAY_LABEL = f"Ridge (lambda*={RIDGE_ALPHA_STAR:.3g})"
LASSO_DISPLAY_LABEL = f"Lasso (lambda*={LASSO_ALPHA_STAR:.3g})"
print(f"Modelos regularizados: {RIDGE_DISPLAY_LABEL}; {LASSO_DISPLAY_LABEL}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, name, parameter in [
    (axes[0], "Ridge", "ridge__alpha"),
    (axes[1], "Lasso", "lasso__alpha"),
]:
    results = TUNING_RESULTS[name]
    alpha = pd.Series(results[f"param_{parameter}"], dtype=float)
    ax.plot(alpha, -results["mean_train_score"], color=U.MUTED, label="train")
    ax.plot(alpha, -results["mean_test_score"], color=U.ACCENT, label="validation")
    if name == "Ridge":
        ax.set_xscale("symlog", linthresh=1e-2)
    else:
        ax.set_xscale("log")
    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel("RMSE")
    ax.set_title(f"{name}: complejidad frente a error", loc="left", fontsize=10)
axes[0].legend(fontsize=9)
U.save_fig(fig, "B04_curvas_regularizacion")
plt.show()

rf_results = TUNING_RESULTS["RandomForest"].copy()
rf_table = pd.DataFrame({
    "max_depth": rf_results["param_max_depth"],
    "min_samples_leaf": rf_results["param_min_samples_leaf"],
    "max_features": rf_results["param_max_features"],
    "RMSE_validation": -rf_results["mean_test_score"],
}).sort_values("RMSE_validation")
display(rf_table.round(3))

# ¿Cuánto contrajo cada regularizador la biblioteca de features?
LINEAR_COEFFICIENTS = {}
regularization_rows = []
for name in REGULARIZED_MODELS:
    fitted = clone(TUNED_MODELS[name])
    fitted.fit(development[ALL_FEATURES], development["delta_next"])
    coefficients = pd.Series(fitted.steps[-1][1].coef_, index=ALL_FEATURES)
    LINEAR_COEFFICIENTS[name] = coefficients
    active = coefficients.abs() > 1e-8
    regularization_rows.append({
        "modelo": name,
        "coeficientes_no_cero": int(active.sum()),
        "económicos_no_cero": int(active.loc[ECONOMIC_FEATURES].sum()),
        "mediana_|beta|_econ": coefficients.loc[ECONOMIC_FEATURES].abs().median(),
    })
regularization_summary = pd.DataFrame(regularization_rows).set_index("modelo")
display(regularization_summary.round(4))

lasso_selected = LINEAR_COEFFICIENTS["Lasso"]
lasso_selected = lasso_selected[lasso_selected.abs() > 1e-8]
lasso_selection_table = pd.DataFrame({
    "familia": [FEATURE_TO_GROUP[feature] for feature in lasso_selected.index],
    "coeficiente_estandarizado": lasso_selected,
}).sort_values("coeficiente_estandarizado", key=lambda values: values.abs(), ascending=False)
display(lasso_selection_table.round(4))

In [ ]:
MODELS = {
    "RW": None,
    "AR(3)-OLS": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": TUNED_MODELS["Ridge"],
    "Lasso": TUNED_MODELS["Lasso"],
    "RandomForest": TUNED_MODELS["RandomForest"],
}
MODEL_FEATURES = {
    "RW": [],
    "AR(3)-OLS": AR_FEATURES,
    "Ridge": ALL_FEATURES,
    "Lasso": ALL_FEATURES,
    "RandomForest": ALL_FEATURES,
}
MODEL_TARGET = {
    "RW": None,
    "AR(3)-OLS": "y_next",
    "Ridge": "delta_next",
    "Lasso": "delta_next",
    "RandomForest": "delta_next",
}

frozen_registry = pd.DataFrame({
    "features": {name: len(MODEL_FEATURES[name]) for name in MODEL_ORDER},
    "objetivo_modelado": {
        name: ("RW: corrección cero" if name == "RW"
               else "nivel y_next" if MODEL_TARGET[name] == "y_next"
               else "corrección delta_next")
        for name in MODEL_ORDER
    },
    "es_modelo_principal": {name: name == PRIMARY_MODEL for name in MODEL_ORDER},
    "configuración": {
        "RW": "sin hiperparámetros",
        "AR(3)-OLS": "3 rezagos, OLS",
        "Ridge": str(BEST_PARAMS["Ridge"]),
        "Lasso": str(BEST_PARAMS["Lasso"]),
        "RandomForest": str(BEST_PARAMS["RandomForest"]),
    },
})
display(frozen_registry)
print(f"Modelo principal predefinido para DM e interpretación: {PRIMARY_MODEL}")

## 6. Test final: backtest expansivo mes a mes

Ahora abrimos el bloque post-2020. En cada origen:

1. entrenamos solo con filas anteriores al origen;
2. pronosticamos el mes siguiente;
3. avanzamos un mes y repetimos.

Las features, los hiperparámetros y el registry siguen congelados. Solo se actualizan los coeficientes o árboles con información que ya pasó a estar disponible. Ridge, Lasso y RF producen una corrección; el código suma `y_t` para regresar al nivel de inflación antes de calcular cualquier métrica.

In [ ]:
from tqdm.auto import tqdm


def walk_forward(panel, test_origins):
    """Backtest con ventana expansiva y decisiones ya congeladas."""
    rows = []

    for origin in tqdm(test_origins, desc="Backtest expansivo"):
        # Solo entran filas cuyo label ya se conoce en este origen.
        train = panel[panel["target_date"] <= origin]
        assert train["target_date"].max() <= origin
        current = panel.loc[[origin]]

        anchor_rw = float(current["y_t"].iloc[0])
        for name in MODEL_ORDER:
            if name == "RW":
                delta_hat = 0.0
                prediction = anchor_rw
            else:
                features = MODEL_FEATURES[name]
                target = MODEL_TARGET[name]
                model = clone(MODELS[name])
                model.fit(train[features], train[target])
                model_output = float(model.predict(current[features])[0])
                if target == "delta_next":
                    delta_hat = model_output
                    prediction = anchor_rw + delta_hat
                else:
                    prediction = model_output
                    delta_hat = prediction - anchor_rw

            rows.append({
                "origin": origin,
                "release_date": current["release_date"].iloc[0],
                "target_date": current["target_date"].iloc[0],
                "model": name,
                "y_true": float(current["y_next"].iloc[0]),
                "y_hat": prediction,
                "delta_hat": delta_hat,
            })

    result = pd.DataFrame(rows)
    result["error"] = result["y_true"] - result["y_hat"]
    return result


backtest = walk_forward(panel, test.index)
print(f"Test ejecutado: {backtest.target_date.min():%Y-%m} a {backtest.target_date.max():%Y-%m}")
print(f"Pronósticos: {len(test)} meses x {len(MODEL_ORDER)} modelos = {len(backtest)}")

## 7. Métricas y evidencia, no solo un ranking

Reportamos RMSE, MAE y sesgo del pronóstico final en puntos porcentuales. Aunque tres modelos aprendieron una corrección, todas las métricas se calculan después de reconstruir el nivel. `relativo_RW` menor que uno indica una mejora frente al benchmark.

El test completo es el resultado principal. Las ventanas 2021-2023 y 2024-2026 son diagnósticos predefinidos para distinguir el episodio inflacionario post-COVID de la normalización posterior.

Distinguiremos dos conceptos. Ridge es el **modelo principal predefinido** antes del test y se contrasta frente al RW con Diebold-Mariano. El modelo con menor RMSE en el test completo es el **ganador descriptivo**: resume lo ocurrido, pero fue identificado después de observar el test. Un p-valor bajo indica diferencia y el signo del estadístico dice quién es más preciso.

In [ ]:
metric_rows = []
for period, (start, end) in SUBPERIODS_PRESET.items():
    end = backtest.target_date.max() if end is None else end
    sample = backtest[backtest["target_date"].between(start, end)]
    rmse_rw = U.rmse(sample.loc[sample["model"] == "RW", "error"])

    for name in MODEL_ORDER:
        errors = sample.loc[sample["model"] == name, "error"]
        rmse = U.rmse(errors)
        metric_rows.append({
            "periodo": period, "modelo": name,
            "RMSE": rmse, "MAE": errors.abs().mean(),
            "sesgo": errors.mean(), "relativo_RW": rmse / rmse_rw,
        })

test_metrics = pd.DataFrame(metric_rows).set_index(["periodo", "modelo"])
display(test_metrics.round(3))

full_test_metrics = test_metrics.loc["Test completo"].sort_values("RMSE")
DESCRIPTIVE_WINNER = full_test_metrics.index[0]
print(f"Modelo principal predefinido: {PRIMARY_MODEL}")
print(f"Ganador descriptivo del test completo: {DESCRIPTIVE_WINNER}")

error_matrix = backtest.pivot(index="target_date", columns="model", values="error")
dm_stat, dm_pvalue = U.dm_test(error_matrix[PRIMARY_MODEL], error_matrix["RW"], h=HORIZON)
winner_text = PRIMARY_MODEL if dm_stat < 0 else "RW"
print(f"DM {PRIMARY_MODEL} vs RW: estadístico = {dm_stat:.2f}, p-valor = {dm_pvalue:.3f}")
print(f"El signo favorece a {winner_text}; la significancia se evalúa con el p-valor.")

In [ ]:
forecast_matrix = backtest.pivot(index="target_date", columns="model", values="y_hat")
actual = backtest.drop_duplicates("target_date").set_index("target_date")["y_true"]
cumulative_gain = (
    error_matrix["RW"].pow(2) - error_matrix[PRIMARY_MODEL].pow(2)
).cumsum()

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
ax = axes[0]
ax.plot(actual.index, actual, color=U.INK, lw=1.7, label="inflación observada")
ax.plot(forecast_matrix.index, forecast_matrix["RW"], color=U.MUTED,
        lw=1.2, ls="dotted", label="RW")
ax.plot(forecast_matrix.index, forecast_matrix[PRIMARY_MODEL], color=U.ACCENT,
        lw=1.4, ls="dashed", label=f"{RIDGE_DISPLAY_LABEL}")
#ax.plot(forecast_matrix.index, forecast_matrix["Lasso"], color=U.BLUE,
#        lw=1.1, ls="dashdot", label=LASSO_DISPLAY_LABEL)
ax.axhspan(1, 3, color=U.OLIVE, alpha=0.10, label="rango meta")
ax.set_ylabel("inflación, % a/a")
ax.set_title("Test post-2020: realizado y pronósticos a un mes", loc="left")
ax.legend(fontsize=9, ncol=1, loc='upper right')

ax = axes[1]
ax.plot(cumulative_gain.index, cumulative_gain, color=U.BLUE, lw=1.6)
ax.axhline(0, color=U.MUTED, lw=0.8)
ax.set_ylabel("ganancia acumulada")
ax.set_xlabel("fecha del objetivo")
ax.set_title(f"Pérdida cuadrática acumulada: positivo favorece a {PRIMARY_MODEL}",
             loc="left", fontsize=10)
U.save_fig(fig, "B05_pronosticos_test_ridge")
plt.show()

## 8. Interpretabilidad de la familia Ridge

Ridge fue declarado como modelo principal antes de abrir el test. Validation seleccionó una penalización estrictamente positiva, por lo que ahora sí hay contracción de una biblioteca amplia y redundante.

Un modelo lineal permite dos lecturas distintas:

- **Coeficiente estandarizado**: cambio en la corrección al RW, y por tanto en el pronóstico final, ante una desviación estándar adicional de la feature ya transformada.
- **Contribución**: coeficiente por valor estandarizado en un origen concreto. Intercepto y contribuciones suman la corrección; al añadir `y_t` obtenemos el pronóstico final.

Interpretaremos el pronóstico para julio de 2022, dentro del episodio de alta inflación. Es una explicación predictiva, no una estimación causal. Mostraremos las doce variables con mayor coeficiente estandarizado y una descomposición exacta de la corrección. Como varias features están correlacionadas, coeficientes y contribuciones dependen de la parametrización.

In [ ]:
def pretty_feature(feature):
    if feature.startswith("delta_y_lag"):
        lag = feature.removeprefix("delta_y_lag")
        return f"cambio inflación a/a, rezago {lag}"
    if feature.startswith("ipc_mom_lag"):
        lag = feature.removeprefix("ipc_mom_lag")
        return f"inflación mensual, rezago {lag}"
    if feature == "ipc_mom_ma3_ann":
        return "inflación mensual, media 3m anualizada"
    if feature == "ipc_mom_ma6_ann":
        return "inflación mensual, media 6m anualizada"
    suffix_labels = {"latest": "último dato", "ma3": "media 3m", "delta3": "cambio 3m"}
    for column in DRIVER_COLUMNS:
        prefix = f"{column}_"
        if feature.startswith(prefix):
            return f"{SERIES_LABELS[column]}: {suffix_labels[feature.removeprefix(prefix)]}"
    return feature

EXPLAIN_TARGET_DATE = pd.Timestamp("2022-07-01")
explain_origin = panel.index[panel["target_date"] == EXPLAIN_TARGET_DATE][0]
explain_train = panel[panel["target_date"] <= explain_origin]
assert explain_train["target_date"].max() <= explain_origin
explain_row = panel.loc[[explain_origin]]
explain_features = MODEL_FEATURES[PRIMARY_MODEL]

explain_model = clone(MODELS[PRIMARY_MODEL])
explain_model.fit(explain_train[explain_features], explain_train["delta_next"])

scaler = explain_model.named_steps["standardscaler"]
linear_model = explain_model.steps[-1][1]
x_standardized = scaler.transform(explain_row[explain_features])[0]
coefficients = np.asarray(linear_model.coef_, dtype=float)
contributions = coefficients * x_standardized
intercept = float(linear_model.intercept_)
delta_prediction = float(explain_model.predict(explain_row[explain_features])[0])
anchor_rw = float(explain_row["y_t"].iloc[0])
prediction = anchor_rw + delta_prediction

assert np.isclose(delta_prediction, intercept + contributions.sum())
assert np.isclose(prediction, anchor_rw + intercept + contributions.sum())
stored_prediction = backtest.loc[
    (backtest["target_date"] == EXPLAIN_TARGET_DATE)
    & (backtest["model"] == PRIMARY_MODEL), "y_hat"
].iloc[0]
assert np.isclose(prediction, stored_prediction)

interpretation = pd.DataFrame({
    "feature": explain_features,
    "familia": [FEATURE_TO_GROUP[f] for f in explain_features],
    "etiqueta": [pretty_feature(f) for f in explain_features],
    "valor_estandarizado": x_standardized,
    "coeficiente_estandarizado": coefficients,
    "contribución": contributions,
}).set_index("feature")
interpretation["importancia_abs"] = interpretation["coeficiente_estandarizado"].abs()
TOP_COEFFICIENTS = interpretation.nlargest(12, "importancia_abs").index.tolist()
TOP_CONTRIBUTIONS = interpretation["contribución"].abs().nlargest(8).index.tolist()
rest_contribution = interpretation.drop(index=TOP_CONTRIBUTIONS)["contribución"].sum()
correction_parts = pd.Series(
    {"intercepto": intercept}
    | {interpretation.loc[f, "etiqueta"]: interpretation.loc[f, "contribución"]
       for f in TOP_CONTRIBUTIONS}
    | {"resto de features": rest_contribution}
)
assert np.isclose(correction_parts.sum(), delta_prediction)

print(f"Origen: {explain_origin:%Y-%m} | Objetivo: {EXPLAIN_TARGET_DATE:%Y-%m}")
print(f"Modelo: {RIDGE_DISPLAY_LABEL}")
print(f"Ancla RW y_t: {anchor_rw:.3f} | Corrección estimada: {delta_prediction:.3f}")
print(f"Intercepto: {intercept:.3f} | Suma de contribuciones: {contributions.sum():.3f}")
print(f"Pronóstico final: {prediction:.3f} | "
      f"Realizado: {float(explain_row.y_next.iloc[0]):.3f}")
display(interpretation.sort_values("importancia_abs", ascending=False).head(15).round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
coef_plot = interpretation.loc[TOP_COEFFICIENTS].sort_values("coeficiente_estandarizado")
axes[0].barh(coef_plot["etiqueta"], coef_plot["coeficiente_estandarizado"],
             color=U.ACCENT, edgecolor=U.BORDER)
axes[0].axvline(0, color=U.MUTED, lw=0.8)
axes[0].set_title(f"Coeficientes estandarizados: {RIDGE_DISPLAY_LABEL}",
                  loc="left", fontsize=10)
axes[0].set_xlabel("puntos porcentuales por 1 DE de la feature transformada")

contrib_plot = correction_parts.sort_values()
axes[1].barh(contrib_plot.index, contrib_plot,
             color=[U.BLUE if value >= 0 else U.ACCENT
                    for value in contrib_plot])
axes[1].axvline(0, color=U.MUTED, lw=0.8)
axes[1].set_title("Contribuciones a la corrección, julio de 2022",
                  loc="left", fontsize=10)
axes[1].set_xlabel("puntos porcentuales; las barras suman la corrección")
U.save_fig(fig, "B06_interpretabilidad_ridge")
plt.show()

## 9. El pipeline completo

```text
pregunta y origen de información
  -> 41 features económicas y deliberadamente redundantes
  -> lags de publicación antes de transformar
  -> test bloqueado desde 2021
  -> challengers pronostican una corrección al RW
  -> escalado dentro del Pipeline
  -> hiperparámetros de Ridge, Lasso y RF con validation 2016-2020
  -> registry de cinco modelos congelado antes del test
  -> backtest expansivo post-2020
  -> RMSE + MAE + sesgo para todos
  -> DM predefinido: Ridge frente al RW
  -> contracción y sparsity de los modelos regularizados
  -> coeficientes y contribuciones de Ridge a una corrección
```

Ojo con una distinción importante: Ridge es el modelo principal fijado antes del test. El ganador descriptivo es simplemente el menor RMSE observado en el test completo. Si ambos coinciden, eso no elimina la diferencia conceptual entre una hipótesis predefinida y un resumen ex post.

## 10. Lo que se lleva a casa

1. **Información comparable**: si el RW observa \(y_t\), los challengers deben compartir esa ancla. Modelar la corrección hace explícita la comparación.
2. **El calendario define las features**: un valor económico que aún no fue publicado no existe para el pronosticador.
3. **El test se bloquea primero**: seleccionar features o hiperparámetros con resultados post-2020 contaminaría la evaluación.
4. **El benchmark siempre corre**: RMSE absoluto sin RW no permite saber si el modelo agrega valor.
5. **Development y test cumplen funciones distintas**: development configura los modelos; el test mide y describe su desempeño posterior.
6. **El ranking puede cambiar por régimen**: por eso reportamos el test completo y subperiodos definidos de antemano.
7. **Regularizar no descubre la verdad**: Ridge contrae señales redundantes; Lasso puede llevar varias a cero, pero ninguna de las dos operaciones identifica causalidad.
8. **Interpretar es reconstruir la predicción**: el ancla RW, el intercepto y las contribuciones explican exactamente el pronóstico de Ridge.

### Para practicar

- Repite el test con una ventana móvil de 10 años, sin cambiar el periodo de test.
- Agrega un feature nuevo con su fecha de publicación y vuelve a configurar los modelos usando solo development.
- Compara los cinco modelos a horizontes de 3 y 6 meses. La idea de que los drivers necesitan tiempo para operar debe probarse, no suponerse.
- Contamina deliberadamente el escalado con toda la muestra y mide cuánto cambia el resultado. Leakage no garantiza que el RMSE mejore, pero invalida la comparación.